# Primitive Rateless: an interactive soft-XOR laboratory

The default sparse polynomial is **$P(x)=x^5+x^2+1$**, with **$K=5$** and **$N=16$**. Polynomial exponents are the only code definition: change `P` below, and `K` follows automatically. We trust your choice of a primitive polynomial; no period enumeration or primality/primitivity check is performed.

The workflow has three independent stages:

1. Run the setup and editor cells, then edit the observation for as long as you like.
2. Run **Decode the current observation** to copy the current inputs and perform $T$ iterations.
3. Run the plot cells to inspect that completed result.

Install with `uv sync --group dev` from the repository root and use the project's Python kernel in VS Code or JupyterLab.


In [1]:
import sys
from pathlib import Path

notebooks_dir = (
    Path.cwd() if (Path.cwd() / "_shared").is_dir() else Path.cwd() / "notebooks"
)
if not (notebooks_dir / "_shared").is_dir():
    raise RuntimeError("Start the kernel in the repository root or notebooks directory")
if str(notebooks_dir.resolve()) not in sys.path:
    sys.path.insert(0, str(notebooks_dir.resolve()))

from _shared.lfsr import polynomial_terms  # noqa: E402
from _shared.widgets import LLREditor  # noqa: E402
from IPython.display import display  # noqa: E402

P = polynomial_terms((0, 2, 5))  # Distinct nonzero exponents, including 0 and K.
# For a degree-32 experiment, supply your chosen degree-32 polynomial here.
K = P[-1]
N = 16  # Choose N > K if you want local parity checks and redundancy.
T = 20
SEED = None  # Use an integer to reproduce the random word and subsequent noise draws.

## Edit the observation $y'$

- Click a bit of $x$ to flip it, or **New random x** to sample a new word. Every change of $x$ resets $y'$ to the newly encoded reference. Changing $N$ also re-encodes and resets; changing $T$ preserves the observation.
- Hold the left mouse button and move over $y'$ cells. A brush acts immediately on entry and repeats every **100 ms** while held over that cell. Keyboard users can focus a cell and press Space or Enter.
- **Flip:** subtract $a s_i$ per tick. This gradually crosses zero; it is not instantaneous sign negation.
- **Suppress:** multiply by $10^{-d/20}$ per tick. Here dB is explicitly an **amplitude attenuation convention for the LLR**, not an SNR conversion. Infinite dB erases immediately.
- **Restore:** add $a s_i$ per tick. It can increase confidence above one. **Reset** instead sets all cells exactly to $s$.
- **Add noise:** add independent $\mathcal N(0,\sigma^2)$ samples to the current $y'$. Repeated clicks accumulate noise. The global control sets $\sigma$ directly in LLR units (default: 0.5 LLR, step: 0.1, minimum: 0).

Green means the sign agrees with the reference; red means it disagrees. Zero is white. Color strength is $1-\exp(-|L|/2)$, on a fixed scale shared with the evolution plot. Hover for the full numeric value.

There is no fixed upper cap on $N$ or $T$. Large words still require rendering and transmitting $N$ editable cells, so the browser can become the bottleneck. Changing $T$ only sets the next explicit run length; it never starts a computation.

In [2]:
# Recreating the editor resets the experiment input; it does not decode.
if "editor" in globals():
    globals()["editor"].close()
import importlib

LLREditor = importlib.reload(sys.modules[LLREditor.__module__]).LLREditor
editor = LLREditor(n=N, iterations=T, terms=P, seed=SEED)
display(editor)

## Synchronous additive soft-XOR iterations

Let $\mathcal C_i$ be the fully contained polynomial checks containing bit $i$. Start with the edited observation $L^{(0)}=y'$. Each iteration is

$$m_{C\to i}^{(t)}=2\operatorname{atanh}\!\left[\prod_{j\in C\setminus\{i\}}\tanh\!\left(L_j^{(t)}/2\right)\right],$$
$$L_i^{(t+1)}=L_i^{(t)}+\sum_{C\in\mathcal C_i}m_{C\to i}^{(t)}.$$

**Every message reads only the previous iteration.** The decoder never reads the true word. Truth is used only by the editor and diagnostics. A check with an erased neighbor sends zero; a bit with no checks stays unchanged.

The implementation evaluates the tanh soft-XOR through an algebraically equivalent, numerically stable pairwise box-plus identity. This avoids rounding `tanh` to exactly +/-1 at high confidence and does not clip LLRs. It uses no damping, normalization, channel reinjection, or early stopping.

This is the requested additive experiment, **not extrinsic belief propagation**: information is reused through cycles and can reinforce incorrect beliefs. Growing magnitude is not by itself evidence of successful correction. Each explicit decoding run starts a fresh trajectory from a copy of the current observation; decoding never overwrites the editor. Later edits cannot change an existing result.

The encoder, decoder and metric loops use `numba.njit(cache=True)`. The first call may compile a kernel; later calls reuse compiled code. The sparse offsets are runtime data, so changing the degree does not require enumerating states or specializing code to a particular polynomial.

For each check, prefix/suffix box-plus folds compute every leave-one-out opinion in $O(|P|)$ work. All reads still come from the previous row. No `fastmath` is used, preserving the zero/erasure and finite-value semantics.

Regrouping floating-point sums changes rounding slightly. Short trajectories agree with the direct tanh formula to numerical precision; long self-reinforcing trajectories can amplify those differences. The update rule itself is unchanged.


## Decode the current observation

Run this cell **after editing**. It captures one independent snapshot and runs the decoder and diagnostics once. Rerunning only a plot cell reuses this result. The timing includes JIT compilation on the first run.

Choose `SOFTXOR` in the cell below:

- **`"tanh"`**: exact $2\operatorname{atanh}(\prod_j\tanh(L_j/2))$, evaluated with stable box-plus. This is the default.
- **`"normalized-min-sum"`**: $\alpha(\prod_j\operatorname{sign}(L_j))\min_j|L_j|$. `NMS_COEFFICIENT` sets $\alpha\in[0,1]$ and is applied **once per check-to-bit opinion**. A zero neighbor makes the opinion zero. With one neighbor the result is $\alpha L$.
- **`"sqrt-sign"`**: for $m$ neighbors use $\frac12\left(\prod_{j=1}^{m}\operatorname{sign}(L_j)\right)\left(\prod_{j=1}^{m}|L_j|\right)^{1/m}$. Two inputs use a square root, three a cube root, and so on. This is symmetric in all inputs; it is not a fold of binary operations. Any zero input makes the opinion zero. For one input the same formula gives $L/2$. Log magnitudes avoid overflow/underflow of intermediate products. The coefficient setting does not affect this mode.

Every mode uses only the preceding iteration and adds its opinions to the current LLR. All three modes take $O(w)$ work per check for $w$ polynomial terms and run under `njit`. The completed result records `result.softxor` and `result.coefficient`.


In [28]:
import importlib
from time import perf_counter

import _shared.decoding as decoding

# Reload implementation changes without resetting the edited observation.
decoding = importlib.reload(decoding)
SOFTXOR = "tanh"  # "tanh", "normalized-min-sum", or "sqrt-sign"
NMS_COEFFICIENT = 0.8  # Used only by normalized-min-sum; range [0, 1].

snapshot = editor.snapshot()
started = perf_counter()
result = decoding.decode(**snapshot, softxor=SOFTXOR, coefficient=NMS_COEFFICIENT)
print(
    f"Decoded N={result.history.shape[1]}, K={result.terms[-1]}, "
    f"T={result.history.shape[0] - 1}, softxor={result.softxor}"
    + (
        f", coefficient={result.coefficient:g}"
        if result.coefficient is not None
        else ""
    )
    + f" in {perf_counter() - started:.3f} s"
)

Decoded N=16, K=5, T=20, softxor=tanh in 0.017 s


## Iteration versus metrics

All plots include $t=0$ and the following $T$ iterations. All five curves share one plot and their original numerical scale. The rate metrics are:

- **Sign error rate:** fraction of negative signed margins, with exact zero counted as half an error.
- **Erasure fraction:** fraction of exactly zero LLRs.
- **Unsatisfied checks:** hard-decision syndrome fraction (zero LLR is decoded as bit 0). This metric is undefined, shown as a gap, if no checks exist. Another valid codeword can have zero syndrome and still be wrong.

The same plot also shows **mean logistic loss** $\frac1N\sum_i\log(1+e^{-s_iL_i})$, sensitive to confidently wrong decisions, and **mean signed margin** $\frac1N\sum_i s_iL_i$, measuring average alignment and confidence. The latter can hide individual failures; inspect the per-bit plot too.

Drag to zoom, use the mouse wheel to zoom, choose pan in the toolbar, and double-click to reset axes. Click legend entries to hide traces; double-click one to isolate it. Editing the widget does not touch these plots. Rerunning a plot cell creates a fresh interactive figure for `result`.

In [29]:
import importlib

import _shared.plots as plots

# Reload only plotting code; preserve the editor and the completed result.
plots = importlib.reload(plots)
plots.metrics_figure(result).show(config=plots.PLOT_CONFIG)

## Iteration versus bits

One trace per transmitted bit with the following quantity

$$q_i^{(t)}=(-1)^{y_i} L_i^{(t)}=(1-2y_i)L_i^{(t)}.$$

Positive is correct, negative is wrong, and zero is erased. Use the legend to inspect individual bits, especially near the boundaries.

In [30]:
import importlib

import _shared.plots as plots

# Reload only plotting code; preserve the editor and the completed result.
plots = importlib.reload(plots)
plots.bits_figure(result).show(config=plots.PLOT_CONFIG)

## Evolution of $y'$

Each row is the complete LLR vector at the next iteration, starting at $t=0$ at the top. Color uses the same truth-relative mapping as the editor. Hover reveals the **raw signed LLR**, not the color-transformed value. The fixed saturating color scale makes low-confidence details visible even when later iterations become very confident.

In [31]:
import importlib

import _shared.plots as plots

# Reload only plotting code; preserve the editor and the completed result.
plots = importlib.reload(plots)
plots.evolution_figure(result).show(config=plots.PLOT_CONFIG)